# (a) Learning Rate Sensitivity Analysis

In [ ]:
#!/usr/bin/env python3
"""
Human-only Learning Rate Sweep
- Trains ShuffleNetV2 on human dataset for different learning rates
- Reports Accuracy and Macro-F1 on validation set
- Saves LR sensitivity plot
"""

import os, random, numpy as np
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ============ CONFIG ============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"  # <-- change this
SAVE_ROOT = Path("/kaggle/working/"); SAVE_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 32, 8
VAL_FRAC, TEST_FRAC = 0.2, 0.2
LR_LIST = [1e-2, 2e-2, 1e-3,2e-3,1e-4,3e-4, 5e-4, 5e-5]  # extend if you want
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============ DATA ============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
NUM_CLASSES = len(base.classes)

def stratified_split_indices(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

h_tr_idx, h_va_idx, h_te_idx = stratified_split_indices(base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)
human_test  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_te_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(human_train, shuffle=True)
val_loader   = make_loader(human_val, shuffle=False)

# ============ TRAIN / EVAL ============
def eval_metrics(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(DEVICE)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100
    f1m = f1_score(y_true, y_pred, average="macro")*100
    return acc, f1m

results = []
for lr in LR_LIST:
    print(f"\n=== Training with LR={lr} ===")
    base_model = models.shufflenet_v2_x0_5(weights="DEFAULT")
    in_f = base_model.fc.in_features
    base_model.fc = nn.Linear(in_f, NUM_CLASSES)
    model = base_model.to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)

    # simple training loop
    for ep in range(1, EPOCHS+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        v_acc, v_f1 = eval_metrics(model, val_loader)
        print(f"Epoch {ep}/{EPOCHS}: val_acc={v_acc:.2f} f1={v_f1:.2f}")

    v_acc, v_f1 = eval_metrics(model, val_loader)
    results.append((lr, v_acc, v_f1))

# ============ PLOT ============
lrs = [r[0] for r in results]
accs = [r[1] for r in results]
f1ms = [r[2] for r in results]

plt.figure(figsize=(6,4.5))
plt.plot(lrs, accs, marker='o', label="Accuracy")
plt.plot(lrs, f1ms, marker='o', label="Macro F1-Score")
plt.xscale("log")
plt.xlabel("Learning Rate")
plt.ylabel("Performance (%)")
plt.title("Learning Rate Sensitivity Analysis")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_ROOT/"lr_sensitivity.png", dpi=200)
print(f"\nPlot saved to {SAVE_ROOT/'lr_sensitivity.png'}")



# Scheduler Strategy Comparison

In [ ]:
#!/usr/bin/env python3
"""
Scheduler Strategy Comparison (5 schedulers)
- Trains ShuffleNetV2 on human dataset
- Applies 5 schedulers: CosineAnnealing, Plateau, Step, Exponential, OneCycle
- Reports Accuracy and Macro-F1
- Saves comparison bar chart
"""

import os, random, numpy as np
from pathlib import Path
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ============ CONFIG ============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata" 
SAVE_ROOT = Path("/kaggle/working/outputs"); SAVE_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 32, 8
VAL_FRAC, TEST_FRAC = 0.2, 0.2
BASE_LR, WEIGHT_DECAY = 1e-3, 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============ DATA ============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
NUM_CLASSES = len(base.classes)

def stratified_split_indices(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

h_tr_idx, h_va_idx, h_te_idx = stratified_split_indices(base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)
human_test  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_te_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(human_train, shuffle=True)
val_loader   = make_loader(human_val, shuffle=False)

# ============ EVAL ============
def eval_metrics(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(DEVICE)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100
    f1m = f1_score(y_true, y_pred, average="macro")*100
    return acc, f1m

# ============ Schedulers ============
def get_scheduler(name, opt):
    if name == "CosineAnnealing":
        return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    elif name == "Plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
    elif name == "StepLR":
        return torch.optim.lr_scheduler.StepLR(opt, step_size=3, gamma=0.5)
    elif name == "Exponential":
        return torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9)
    elif name == "OneCycle":
        return torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=BASE_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS)
    else:
        raise ValueError("Unknown scheduler")

SCHEDULERS = ["CosineAnnealing", "Plateau", "StepLR", "Exponential", "OneCycle"]

# ============ TRAIN LOOP ============
ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
results = []

for sched_name in SCHEDULERS:
    print(f"\n=== Training with Scheduler: {sched_name} ===")
    base_model = models.shufflenet_v2_x0_5(weights="DEFAULT")
    in_f = base_model.fc.in_features
    base_model.fc = nn.Linear(in_f, NUM_CLASSES)
    model = base_model.to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    sched = get_scheduler(sched_name, opt)

    for ep in range(1, EPOCHS+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
            if sched_name == "OneCycle": sched.step()
        v_acc, v_f1 = eval_metrics(model, val_loader)
        print(f"Epoch {ep}/{EPOCHS}: val_acc={v_acc:.2f} f1={v_f1:.2f}")
        if sched_name == "Plateau": sched.step(v_acc)
        elif sched_name != "OneCycle": sched.step()

    v_acc, v_f1 = eval_metrics(model, val_loader)
    results.append((sched_name, v_acc, v_f1))

# ============ PLOT ============
labels = [r[0] for r in results]
accs   = [r[1] for r in results]
f1ms   = [r[2] for r in results]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(7,5))
plt.bar(x - width/2, accs, width, label="Accuracy")
plt.bar(x + width/2, f1ms, width, label="Macro F1-Score")
plt.xticks(x, labels, rotation=20)
plt.ylabel("Performance (%)")
plt.title("Scheduler Strategy Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_ROOT/"scheduler_comparison.png", dpi=200)
print(f"\nPlot saved to {SAVE_ROOT/'scheduler_comparison.png'}")


# Optimizers

In [ ]:
#!/usr/bin/env python3
"""
Optimizer Performance Comparison (5 optimizers)
- Trains ShuffleNetV2 on human dataset
- Applies 5 optimizers: Adam, AdamW, SGD, RMSprop, Adagrad
- Reports Accuracy and Macro-F1
- Saves comparison bar chart
"""

import os, random, numpy as np
from pathlib import Path
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ============ CONFIG ============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"
SAVE_ROOT = Path("/kaggle/working/outputs"); SAVE_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 32, 4
VAL_FRAC, TEST_FRAC = 0.2, 0.2
LR, WEIGHT_DECAY = 1e-3, 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============ DATA ============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
NUM_CLASSES = len(base.classes)

def stratified_split_indices(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

h_tr_idx, h_va_idx, h_te_idx = stratified_split_indices(base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(human_train, shuffle=True)
val_loader   = make_loader(human_val, shuffle=False)

# ============ EVAL ============
def eval_metrics(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(DEVICE)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100
    f1m = f1_score(y_true, y_pred, average="macro")*100
    return acc, f1m

# ============ Optimizers ============
def get_optimizer(name, params):
    if name == "Adam":     return torch.optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)
    if name == "AdamW":    return torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    if name == "SGD":      return torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
    if name == "RMSprop":  return torch.optim.RMSprop(params, lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
    if name == "Adagrad":  return torch.optim.Adagrad(params, lr=LR, weight_decay=WEIGHT_DECAY)
    raise ValueError("Unknown optimizer")

OPTIMIZERS = ["Adam", "AdamW", "SGD", "RMSprop", "Adagrad"]

# ============ TRAIN LOOP ============
ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
results = []

for opt_name in OPTIMIZERS:
    print(f"\n=== Training with Optimizer: {opt_name} ===")
    base_model = models.shufflenet_v2_x0_5(weights="DEFAULT")
    in_f = base_model.fc.in_features
    base_model.fc = nn.Linear(in_f, NUM_CLASSES)
    model = base_model.to(DEVICE)

    opt = get_optimizer(opt_name, model.parameters())

    for ep in range(1, EPOCHS+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        v_acc, v_f1 = eval_metrics(model, val_loader)
        print(f"Epoch {ep}/{EPOCHS}: val_acc={v_acc:.2f} f1={v_f1:.2f}")

    v_acc, v_f1 = eval_metrics(model, val_loader)
    results.append((opt_name, v_acc, v_f1))

# ============ PLOT ============
labels = [r[0] for r in results]
accs   = [r[1] for r in results]
f1ms   = [r[2] for r in results]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(7,5))
bars1 = plt.bar(x - width/2, accs, width, label="Accuracy", color="Blue", edgecolor="black")
bars2 = plt.bar(x + width/2, f1ms, width, label="Macro F1-Score", color="red", edgecolor="black")
plt.xticks(x, labels, rotation=20)
plt.ylabel("Performance (%)")
plt.title("Optimizer Performance Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_ROOT/"optimizer_comparison.png", dpi=200)
print(f"\nPlot saved to {SAVE_ROOT/'optimizer_comparison.png'}")


# Weight Decay

In [ ]:
#!/usr/bin/env python3
"""
Weight Decay Regularization Analysis
- Trains ShuffleNetV2 on human dataset
- Sweeps different weight decay values
- Reports Accuracy and Macro-F1
- Plots line chart (log scale on x-axis)
"""

import os, random, numpy as np
from pathlib import Path
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ============ CONFIG ============
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

HUMAN_IMG_PATH = "/kaggle/input/human-reticulocyte/SubsetAlldata"  # <-- fixed path
SAVE_ROOT = Path("/kaggle/working/outputs"); SAVE_ROOT.mkdir(parents=True, exist_ok=True)

IMG_SIZE, BATCH_SIZE, EPOCHS = 224, 20, 5
VAL_FRAC, TEST_FRAC = 0.2, 0.2
LR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Weight decay sweep
WEIGHT_DECAYS = [1e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

# ============ DATA ============
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

base = datasets.ImageFolder(HUMAN_IMG_PATH, transform=None)
NUM_CLASSES = len(base.classes)

def stratified_split_indices(base_ds, val_frac, test_frac=0.0, seed=SEED):
    rng = np.random.default_rng(seed)
    targets = np.array([y for _, y in base_ds.samples])
    idx = np.arange(len(targets))
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(targets):
        ci = idx[targets==c].copy(); rng.shuffle(ci)
        n = len(ci)
        n_val  = max(1, int(round(n*val_frac)))
        n_test = max(0, int(round(n*test_frac)))
        val_idx.extend(ci[:n_val])
        test_idx.extend(ci[n_val:n_val+n_test])
        train_idx.extend(ci[n_val+n_test:])
    return train_idx, val_idx, test_idx

h_tr_idx, h_va_idx, _ = stratified_split_indices(base, VAL_FRAC, TEST_FRAC)
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), h_tr_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf),  h_va_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

train_loader = make_loader(human_train, shuffle=True)
val_loader   = make_loader(human_val, shuffle=False)

# ============ EVAL ============
def eval_metrics(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(DEVICE)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100
    f1m = f1_score(y_true, y_pred, average="macro")*100
    return acc, f1m

# ============ TRAIN LOOP ============
ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
results = []

for wd in WEIGHT_DECAYS:
    print(f"\n=== Training with Weight Decay={wd} ===")
    base_model = models.shufflenet_v2_x0_5(weights="DEFAULT")
    in_f = base_model.fc.in_features
    base_model.fc = nn.Linear(in_f, NUM_CLASSES)
    model = base_model.to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=wd)

    for ep in range(1, EPOCHS+1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb)
            loss = ce_loss(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            opt.step()
        v_acc, v_f1 = eval_metrics(model, val_loader)
        print(f"Epoch {ep}/{EPOCHS}: val_acc={v_acc:.2f} f1={v_f1:.2f}")

    v_acc, v_f1 = eval_metrics(model, val_loader)
    results.append((wd, v_acc, v_f1))

# ============ PLOT ============
wds   = [r[0] for r in results]
accs  = [r[1] for r in results]
f1ms  = [r[2] for r in results]

plt.figure(figsize=(7,5))
plt.plot(wds, accs, marker='o', color="royalblue", label="Accuracy", linewidth=2, markeredgecolor="black")
plt.plot(wds, f1ms, marker='s', color="darkorange", label="Macro F1-Score", linewidth=2, markeredgecolor="black")
plt.xscale("log")
plt.xlabel("Weight Decay", fontsize=12)
plt.ylabel("Performance (%)", fontsize=12)
plt.title("Weight Decay Regularization Analysis", fontsize=14, weight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig(SAVE_ROOT/"weight_decay_analysis.png", dpi=200)
print(f"\nPlot saved to {SAVE_ROOT/'weight_decay_analysis.png'}")


# Voting strategy
✅ The 5 Ensemble Strategies

Soft Voting – average probabilities.

Hard Voting – majority vote on predicted classes.

Accuracy-Weighted Voting – static weight per teacher (your current weighted).

Entropy-Weighted Voting – teachers with lower output entropy (more confident) get more weight.

Per-Sample Confidence Weighting – weight teachers dynamically based on their confidence for each image.

In [ ]:
#!/usr/bin/env python3
# === Teacher Ensemble Voting Strategies (5 methods + Train/Val logging + Augment + Pruning) ===

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt
import torch.nn.utils.prune as prune
import random

# ====================== CONFIG ======================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
IMG_SIZE   = 224
EPOCHS     = 8
PRUNE_AMOUNT = 0.5   # prune 50% of weights (moderate)
HUMAN_IMG_PATH  = "/kaggle/input/human-reticulocyte/SubsetAlldata"

# ====================== DATA AUGMENT ======================
mean_imnet = [0.485, 0.456, 0.406]
std_imnet  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean_imnet, std_imnet),
])

human_base  = datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf)
NUM_CLASSES = len(human_base.classes)

# Split: 70% train, 15% val, 15% test
indices = np.arange(len(human_base))
np.random.seed(42); np.random.shuffle(indices)
split1 = int(0.7*len(indices))
split2 = int(0.85*len(indices))
train_idx, val_idx, test_idx = indices[:split1], indices[split1:split2], indices[split2:]

# Reapply transforms: train with aug, val/test with eval
human_train = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=train_tf), train_idx)
human_val   = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf), val_idx)
human_test  = Subset(datasets.ImageFolder(HUMAN_IMG_PATH, transform=eval_tf), test_idx)

def make_loader(ds, shuffle=False):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=True)

human_train_loader = make_loader(human_train, shuffle=True)
human_val_loader   = make_loader(human_val, shuffle=False)
human_test_loader  = make_loader(human_test, shuffle=False)

# ====================== UTILS ======================
def replace_classifier_for_num_classes(model, num_classes, name=None):
    if hasattr(model, 'fc'):  # ResNet-style
        in_features = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, num_classes)
        )
    elif name and "squeezenet" in name:
        in_channels = model.classifier[1].in_channels
        model.classifier[1] = nn.Conv2d(in_channels, num_classes, kernel_size=(1,1))
        model.num_classes = num_classes
    else:  # MobileNet/EfficientNet
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, num_classes)
        )
    return model

@torch.no_grad()
def eval_metrics(model, loader, device=DEVICE):
    model.eval()
    y_true, y_pred = [], []
    for x,y in loader:
        x = x.to(device)
        y_true.extend(y.numpy().tolist())
        y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true,y_pred)*100.0
    f1m = f1_score(y_true,y_pred,average="macro")*100.0
    return acc,f1m

# ====================== CUTMIX ======================
def cutmix(x, y, alpha=1.0):
    """Apply CutMix to a batch"""
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    bbx1, bby1, bbx2, bby2 = rand_bbox(x.size(), lam)
    x[:, :, bbx1:bbx2, bby1:bby2] = x[index, :, bbx1:bbx2, bby1:bby2]

    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def rand_bbox(size, lam):
    W = size[2]
    H = size[3]
    cut_rat = np.sqrt(1. - lam)
    cut_w = np.int32(W * cut_rat)
    cut_h = np.int32(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    bbx1 = np.clip(cx - cut_w // 2, 0, W)
    bby1 = np.clip(cy - cut_h // 2, 0, H)
    bbx2 = np.clip(cx + cut_w // 2, 0, W)
    bby2 = np.clip(cy + cut_h // 2, 0, H)

    return bbx1, bby1, bbx2, bby2

# ====================== TRAINING & PRUNING ======================
def train_one_model(model, train_loader, val_loader, epochs=EPOCHS):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for x,y in train_loader:
            x,y = x.to(DEVICE), y.to(DEVICE)

            # --- CutMix 50% probability ---
            if random.random() < 0.5:
                x, y_a, y_b, lam = cutmix(x, y)
                out = model(x)
                loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
            else:
                out = model(x)
                loss = criterion(out,y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        # --- Evaluate train & val ---
        train_acc, train_f1 = eval_metrics(model, train_loader, DEVICE)
        val_acc, val_f1     = eval_metrics(model, val_loader, DEVICE)

        print(f"Epoch {epoch+1}/{epochs} "
              f"Loss={epoch_loss/len(train_loader):.4f} "
              f"TrainAcc={train_acc:.2f} ValAcc={val_acc:.2f} "
              f"TrainF1={train_f1:.2f} ValF1={val_f1:.2f}")
    return model

def prune_model(model, amount=PRUNE_AMOUNT):
    parameters_to_prune = []
    for module in model.modules():
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            parameters_to_prune.append((module, 'weight'))

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    print(f"Pruned {amount*100:.1f}% of weights globally")
    return model

# ====================== TEACHERS ======================
teachers_cfg = [
    ("mobilenet_v2",    models.mobilenet_v2),
    ("efficientnet_b0", models.efficientnet_b0),
    ("squeezenet1_1",   models.squeezenet1_1),
]

teachers = {}
for name, fn in teachers_cfg:
    print(f"\n==== Load Teacher: {name} ====")
    model = fn(weights="DEFAULT")
    model = replace_classifier_for_num_classes(model, NUM_CLASSES, name).to(DEVICE)

    # Train with CutMix + Aug
    model = train_one_model(model, human_train_loader, human_val_loader, epochs=EPOCHS)

    # Moderate pruning + 1 epoch fine-tune
    model = prune_model(model, amount=PRUNE_AMOUNT)
    model = train_one_model(model, human_train_loader, human_val_loader, epochs=1)

    acc,f1m = eval_metrics(model,human_test_loader,DEVICE)
    teachers[name] = {"model":model,"acc":acc,"f1":f1m}
    print(f"[{name}] TEST acc={acc:.2f} f1={f1m:.2f}")

# ====================== ENSEMBLE STRATEGIES ======================
@torch.no_grad()
def ensemble_eval(teachers, loader, device=DEVICE, strategy="soft", gamma=1.5):
    models_list = [t["model"].to(device).eval() for t in teachers.values()]
    accs = [t["acc"] for t in teachers.values()]

    y_true,y_pred = [],[]
    for imgs,labels in loader:
        imgs = imgs.to(device)
        probs_list = [F.softmax(m(imgs),dim=1).cpu().numpy() for m in models_list]

        if strategy=="soft":
            probs = sum(probs_list)/len(probs_list)
            preds = probs.argmax(axis=1)

        elif strategy=="hard":
            preds_stack = np.stack([p.argmax(axis=1) for p in probs_list],axis=0)
            preds = [np.bincount(col).argmax() for col in preds_stack.T]

        elif strategy=="weighted":
            weights = np.array(accs)/sum(accs)
            probs = sum(w*p for w,p in zip(weights,probs_list))
            preds = probs.argmax(axis=1)

        elif strategy=="entropy":
            entropies = [(-p*np.log(p+1e-9)).sum(axis=1).mean() for p in probs_list]
            weights = np.array([accs[i]**gamma/(1+entropies[i]) for i in range(len(accs))])
            weights = weights/weights.sum()
            probs = sum(w*p for w,p in zip(weights,probs_list))
            preds = probs.argmax(axis=1)

        elif strategy=="confidence":
            confs = np.array([p.max(axis=1) for p in probs_list])
            confs = confs / (confs.sum(axis=0, keepdims=True)+1e-9)
            probs = sum(c[:,None]*p for c,p in zip(confs,probs_list))
            preds = probs.argmax(axis=1)

        y_pred.extend(preds); y_true.extend(labels.numpy().tolist())

    acc = accuracy_score(y_true,y_pred)*100.0
    f1m = f1_score(y_true,y_pred,average="macro")*100.0
    return acc,f1m

strategies = ["soft","hard","weighted","entropy","confidence"]
ens_results = {s: ensemble_eval(teachers,human_test_loader,DEVICE,s) for s in strategies}

print("\n=== Ensemble Results ===")
for s,(acc,f1) in ens_results.items():
    print(f"[Ensemble-{s}] TEST acc={acc:.2f} f1={f1:.2f}")

plt.figure(figsize=(8,5))
for s,(acc,f1) in ens_results.items():
    plt.bar(s,acc,label=f"F1={f1:.1f}")
plt.ylabel("Accuracy (%)")
plt.title("Teacher Ensemble Strategies (CutMix + Aug + Pruning)")
plt.legend()
plt.show()


# Temperature

In [ ]:
#!/usr/bin/env python3
"""
Distillation Temperature Sensitivity
- Trains student with KD under different temperatures
- Reports Accuracy & Macro-F1
- Plots sensitivity curve
"""

import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# === CONFIG ===
TEMPS = [1, 2, 5, 7, 10, 15, 20]   # list of temperatures to sweep
ALPHA = 0.5
EPOCHS_STUDENT = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ce_loss = nn.CrossEntropyLoss(label_smoothing=0.05)
kl_loss = nn.KLDivLoss(reduction="batchmean")

# Teachers (already trained from your previous script)
t_models = [t["model"].to(DEVICE).eval() for t in teachers.values()]

def kd_epoch(student, loader, T):
    student.train()
    total, correct, n = 0.0, 0, 0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE), yb.to(DEVICE)
        with torch.no_grad():
            ps = [F.softmax(m(xb)/T, dim=1) for m in t_models]
            t_soft = sum(ps)/len(ps)
        logits = student(xb)
        loss = ALPHA*ce_loss(logits, yb) + (1-ALPHA)*kl_loss(F.log_softmax(logits/T, dim=1), t_soft)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 2.0)
        opt.step()
        total += loss.item()*xb.size(0)
        correct += (logits.argmax(1)==yb).sum().item()
        n += xb.size(0)
    return total/max(n,1), correct/max(n,1)

def eval_metrics(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for x,y in loader:
            x = x.to(DEVICE)
            y_true.extend(y.numpy().tolist())
            y_pred.extend(model(x).argmax(1).cpu().numpy().tolist())
    acc = accuracy_score(y_true, y_pred)*100
    f1m = f1_score(y_true, y_pred, average="macro")*100
    return acc, f1m

results = []
for T in TEMPS:
    print(f"\n=== Training with Temperature T={T} ===")
    # new student
    from torchvision import models
    student = models.shufflenet_v2_x0_5(weights="DEFAULT")
    in_f = student.fc.in_features
    student.fc = nn.Linear(in_f, len(human_train.dataset.classes))
    student = student.to(DEVICE)

    opt = torch.optim.Adam(student.parameters(), lr=1e-4, weight_decay=1e-4)

    for ep in range(1, EPOCHS_STUDENT+1):
        kd_epoch(student, human_train_loader, T)
    acc, f1 = eval_metrics(student, human_val_loader)
    results.append((T, acc, f1))
    print(f"T={T}: acc={acc:.2f} f1={f1:.2f}")

# === PLOT ===
temps = [r[0] for r in results]
accs  = [r[1] for r in results]
f1ms  = [r[2] for r in results]

plt.figure(figsize=(7,5))
plt.plot(temps, accs, 'o-', label="Accuracy")
plt.plot(temps, f1ms, 's-', label="Macro F1-Score")
plt.xlabel("Temperature (T)")
plt.ylabel("Performance (%)")
plt.title("Distillation Temperature Sensitivity")

# highlight optimal point
opt_idx = np.argmax(f1ms)
plt.annotate(f"Optimal T={temps[opt_idx]}",
             xy=(temps[opt_idx], f1ms[opt_idx]),
             xytext=(temps[opt_idx]+2, f1ms[opt_idx]+1),
             arrowprops=dict(facecolor='red', shrink=0.05),
             fontsize=10, color="red")

plt.legend()
plt.tight_layout()
plt.savefig("outputs/distillation_temp_sensitivity.png", dpi=200)
print("Plot saved to outputs/distillation_temp_sensitivity.png")


# ENHANCED PLOTS of previous graph with more precision


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Scheduler names
labels = ["CosineAnnealing", "Plateau", "StepLR", "Exponential", "OneCycle"]

# Final reported values (from your logs)
accs = [98.97, 98.45, 99.29, 98.97, 97.94]
f1ms = [98.95, 98.29, 99.31, 98.96, 97.91]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(8,6))
plt.bar(x - width/2, accs, width, label="Accuracy", color="#4FC3F7",edgecolor="black")
plt.bar(x + width/2, f1ms, width, label="Macro F1-Score", color="#0D47A1",edgecolor="black")



plt.xticks(x, labels, rotation=20, fontsize=11)
plt.ylabel("Performance (%)", fontsize=12)
plt.title("Scheduler Strategy Comparison", fontsize=14, weight="bold")
plt.ylim(97, 100)  # zoom in between 97–100 for visibility

# Add value labels on bars
for i, v in enumerate(accs):
    plt.text(i - width/2, v + 0.05, f"{v:.2f}", ha='center', fontsize=9)
for i, v in enumerate(f1ms):
    plt.text(i + width/2, v + 0.05, f"{v:.2f}", ha='center', fontsize=9)

plt.legend(fontsize=11)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Plot with three shades of blue and black edges for clarity

import matplotlib.pyplot as plt
import numpy as np

# Data from training results (final epoch values)
optimizers = ['Adam', 'AdamW', 'SGD', 'RMSprop', 'Adagrad']
accuracy = [96.91, 97.94, 60.82, 83.51, 89.00]  # Accuracy
f1_score = [96.87, 97.92, 52.17, 83.46, 88.96]  # F1-Score

# Calculate precision difference (accuracy - f1)
precision_diff = np.array(accuracy) - np.array(f1_score)

x = np.arange(len(optimizers))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 6))
fig, ax = plt.subplots(figsize=(8, 6))

rects1 = ax.bar(x - width, accuracy, width, label='Accuracy', color='darkblue', edgecolor='black')
rects2 = ax.bar(x, f1_score, width, label='Macro F1-Score', color='royalblue', edgecolor='black')
rects3 = ax.bar(x + width, precision_diff, width, label='Acc - F1 Difference', color='lightblue', edgecolor='black')

ax.set_ylabel('Performance (%)')
ax.set_title('Optimizer Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(optimizers)
ax.legend()

def add_labels(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, color='black')

add_labels(rects1)
add_labels(rects2)
add_labels(rects3)

plt.tight_layout()
plt.show()

In [ ]:
# Prepare weight decay values and last epoch results
weight_decays = [1e-05, 0.0001, 0.0005, 0.001, 0.005, 0.01]
accuracy = [93.81, 98.63, 98.63, 98.28, 98.97, 97.59]
f1_score = [93.89, 98.63, 98.60, 98.26, 98.97, 97.60]

fig, ax = plt.subplots(figsize=(8, 6))
fig, ax = plt.subplots(figsize=(10, 6))  # wider figure for more spacing

# Plot accuracy and F1 in distinct shades of blue with clear markers
ax.plot(weight_decays, accuracy, marker='o', color='darkblue', label='Accuracy', linewidth=2, markersize=8)
ax.plot(weight_decays, f1_score, marker='s', color='deepskyblue', label='Macro F1-Score', linewidth=2, markersize=8)

# Improve x-axis visibility (still log scale but with more spacing)
ax.set_xscale('log')
ax.set_xticks(weight_decays)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.set_xlim(min(weight_decays) / 2, max(weight_decays) * 2)

# Labels and title
ax.set_xlabel('Weight Decay', fontsize=12)
ax.set_ylabel('Performance (%)', fontsize=12)
ax.set_title('Weight Decay Regularization Analysis', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

# Add value annotations
for x, y in zip(weight_decays, accuracy):
    ax.annotate(f'{y:.2f}', (x, y), textcoords="offset points", xytext=(0, 6), ha='center', fontsize=9, color='darkblue')
for x, y in zip(weight_decays, f1_score):
    ax.annotate(f'{y:.2f}', (x, y), textcoords="offset points", xytext=(0, -12), ha='center', fontsize=9, color='royalblue')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensemble results
strategies = ["Soft Voting", "Hard Voting", "Weighted", "Entropy", "Confidence"]
acc = [95.43, 61.64, 95.43, 95.43, 95.43]
f1 = [95.46, 51.78, 95.46, 95.46, 95.46]

x = np.arange(len(strategies))

# Distinct colors for each strategy pair (darker = accuracy, lighter = f1)
acc_colors = ["#1f77b4", "#2ca02c", "#ff69b4", "#ff7f0e", "#9467bd"]  # blue, green, pink, orange, purple
f1_colors = ["#6baed6", "#98fb98", "#f78fb3", "#ffb347", "#c5b0d5"]   # lighter matching tones

plt.figure(figsize=(9, 6))

# Plot accuracy and F1 bars
bars1 = plt.bar(x - 0.2, acc, width=0.4, color=acc_colors, label="Accuracy")
bars2 = plt.bar(x + 0.2, f1, width=0.4, color=f1_colors, label="F1 Score")

# Add values on top of bars with precision
for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1, f"{height:.2f}%", 
             ha='center', va='bottom', fontsize=9, color="black")
for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 1, f"{height:.2f}%", 
             ha='center', va='bottom', fontsize=9, color="black")

# Labels and styling
plt.xticks(x, strategies, rotation=30, ha="right")
plt.ylabel("Score (%)")
plt.title("Performance vs. Voting Strategy", fontsize=14)
plt.legend()
plt.ylim(0, 110)

# Remove grid
plt.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Distillation temperature sensitivity data
temperatures = [1, 2, 5, 7, 10, 15, 20]
acc = [86.24, 87.61, 89.91, 91.28, 83.49, 89.91, 86.24]
f1 = [86.35, 87.83, 89.97, 91.34, 83.13, 89.95, 86.68]

plt.figure(figsize=(9, 6))

# Plot Accuracy and F1 with smooth colorful lines
plt.plot(temperatures, acc, marker='o', linestyle='-', color="#1f77b4", linewidth=2, label="Accuracy")
plt.plot(temperatures, f1, marker='s', linestyle='--', color="#ff7f0e", linewidth=2, label="F1 Score")

# Highlight max performance point
best_idx = max(range(len(acc)), key=lambda i: acc[i])
plt.scatter(temperatures[best_idx], acc[best_idx], color="blue", s=120, edgecolors="black", zorder=5)
plt.text(temperatures[best_idx], acc[best_idx]+1, f"Best Acc: {acc[best_idx]:.2f}% (T={temperatures[best_idx]})",
         ha="center", fontsize=9, color="black")

best_idx_f1 = max(range(len(f1)), key=lambda i: f1[i])
plt.scatter(temperatures[best_idx_f1], f1[best_idx_f1], color="royalblue", s=120, edgecolors="black", zorder=5)
plt.text(temperatures[best_idx_f1], f1[best_idx_f1]-3, f"Best F1: {f1[best_idx_f1]:.2f}% (T={temperatures[best_idx_f1]})",
         ha="center", fontsize=9, color="black")

# Labels & styling
plt.title("Distillation Sensitivity to Temperature", fontsize=14)
plt.xlabel("Temperature (T)", fontsize=12)
plt.ylabel("Score (%)", fontsize=12)
plt.xticks(temperatures)
plt.ylim(80, 95)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()